# how to use `swarms` module

## Imports

Make sure `swarms.py` is in the current directory.

In [1]:
from datetime import datetime, timezone, timedelta

import pandas as pd
from numpy import random

import swarms

# Limit the number of rows displayed in outputs
# for this tutorial
pd.options.display.max_rows = 5

## Prepare Earthquake Data

Data must be prepared before using the function `find_swarms()`:
* **Data must be in a _pandas DataFrame_**
  * Each row must correspond to one earthquake
  * There must be at least four columns for (1) origin time, (2) latitude, (3) longitude, and (4) magnitude
* **Origin times of earthquakes must be _timezone-aware datetime objects_**
  * Strings must be converted
* **Latitudes and longitudes must be _integers_ or _floats_ representing decimal degrees**
  * Strings must be converted

Let's create some sample data:

In [2]:
latitudes = random.uniform(low=0, high=5, size=200)
longitudes = random.uniform(-25, -20, 200)
mags = random.uniform(4, 5.5, 200)
dates = [datetime(2000, 1, 1, 12, 45, 30, 15, tzinfo=timezone.utc)
         + timedelta(hours=a) for a in random.uniform(0, 100, 200)]
test_df = pd.DataFrame({
    'date': dates,
    'lat': latitudes,
    'lon': longitudes,
    'magnitude': mags
})
test_df

,date,lat,lon,magnitude
0,2000-01-05 14:00:42.991177+00:00,4.109998,-22.404294,5.480244
1,2000-01-02 23:54:53.023650+00:00,3.114578,-23.723602,4.190864
...,...,...,...,...
198,2000-01-01 20:49:24.542082+00:00,0.288400,-21.707160,4.262555
199,2000-01-03 04:13:35.230831+00:00,3.226463,-24.269586,4.553606


We can also read data locally:

In [ ]:
path = "C:/Users/your_name/folder/file.csv"
# For earthquake data downloaded from USGS, reading the CSV with pandas and
# parsing dates is sufficient
data = pd.read_csv(path, parse_dates=[0])
data

## Call Function `find_swarms()`

The function can be called with up to eight arguments, one of which is required (`eq_df`). More information about the arguments (including default values) is available in the docstring at the end of this notebook. Let's call the function using our sample data:

In [3]:
results = swarms.find_swarms(
    eq_df=test_df,        # DataFrame of earthquake data
    time_lim=48,          # Maximum time (hours) between successive swarm earthquakes
    dist_lim=50,          # Maximum distance (km) between an earthquake
                          # and the nearest preceding earthquake in the same swarm
    min_number=3,         # Minimum number of earthquakes required to be considered a swarm
    time_col="date",      # Label for DataFrame column in which values are origin times
    lat_col="lat",        # Label for DataFrame column in which values are latitudes
    lon_col="lon",        # Label for DataFrame column in which values are longitudes
    mag_col="magnitude"   # Label for DataFrame column in which values are magnitudes
)

In [ ]:
# For earthquake data downloaded from USGS:
results = swarms.find_swarms(
    eq_df=data,
    time_lim=48,
    dist_lim=50,
    min_number=3
)

The function returns a named tuple. The first item is a DataFrame of swarms and their properties. The second is a DataFrame of earthquakes, including the ID of the swarm to which they belong.

We can access the contents of the tuple in 3 ways:

### 1. field names "swarms" and "quakes"

In [4]:
results.swarms

,Swarm ID,Start Time,End Time,Duration,Origin Latitude,Origin Longitude,Centre Latitude,Centre Longitude,Number of Earthquakes,Magnitudes,Time Intervals (h),Distance Intervals (km),Minimum Time Limit (h),Minimum Distance Limit (km),Average Time Interval (h),Average Distance Interval (km)
0,0,2000-01-01 13:35:33.740915+00:00,2000-01-05 10:52:01.417453+00:00,3 days 21:16:27.676538,2.796269,-20.866927,2.912540,-20.932767,10,"[4.716180922504579, 4.051831335693045, 4.71988...","[16.051896626666668, 5.807235363333334, 7.2911...","[37.50526709650672, 35.40305074737498, 24.0290...",26.103586,48.041596,10.363817,25.708724
1,1,2000-01-01 13:51:31.070540+00:00,2000-01-05 10:55:48.543430+00:00,3 days 21:04:17.472890,1.879701,-20.350824,1.976687,-20.367298,12,"[5.492676685891994, 4.370262152424703, 4.39392...","[6.709280191111111, 2.0382496900000002, 10.354...","[48.699919361787536, 44.77609028480145, 13.790...",19.315449,48.699919,8.461047,24.323674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,18,2000-01-03 02:29:06.741023+00:00,2000-01-05 01:32:30.067949+00:00,1 days 23:03:23.326926,3.077176,-24.392430,3.290386,-24.419062,5,"[5.149594524414461, 4.553605628962466, 5.08451...","[1.7412471688888889, 21.926795539444445, 10.36...","[21.422988453916386, 42.19822186694033, 19.675...",21.926796,42.198222,11.764120,24.190343
19,19,2000-01-03 03:49:13.509905+00:00,2000-01-04 14:20:43.116179+00:00,1 days 10:31:29.606274,4.573955,-21.768492,4.529423,-21.961956,3,"[4.424323615814165, 4.8063056873675825, 4.2949...","[14.6592333, 19.865657331666664]","[42.895473153531356, 11.088576724041175]",19.865657,42.895473,17.262445,26.992025


In [5]:
results.quakes

,Swarm ID,date,lat,lon,magnitude
0,<NA>,2000-01-01 13:12:52.944104+00:00,2.830076,-24.965474,4.007774
1,0,2000-01-01 13:35:33.740915+00:00,2.796269,-20.866927,4.716181
...,...,...,...,...,...
198,6,2000-01-05 16:20:15.778102+00:00,4.394290,-22.385421,4.407223
199,8,2000-01-05 16:37:38.729424+00:00,0.193753,-21.534493,5.001654


### 2. indexing

In [6]:
results[0]  # swarms

,Swarm ID,Start Time,End Time,Duration,Origin Latitude,Origin Longitude,Centre Latitude,Centre Longitude,Number of Earthquakes,Magnitudes,Time Intervals (h),Distance Intervals (km),Minimum Time Limit (h),Minimum Distance Limit (km),Average Time Interval (h),Average Distance Interval (km)
0,0,2000-01-01 13:35:33.740915+00:00,2000-01-05 10:52:01.417453+00:00,3 days 21:16:27.676538,2.796269,-20.866927,2.912540,-20.932767,10,"[4.716180922504579, 4.051831335693045, 4.71988...","[16.051896626666668, 5.807235363333334, 7.2911...","[37.50526709650672, 35.40305074737498, 24.0290...",26.103586,48.041596,10.363817,25.708724
1,1,2000-01-01 13:51:31.070540+00:00,2000-01-05 10:55:48.543430+00:00,3 days 21:04:17.472890,1.879701,-20.350824,1.976687,-20.367298,12,"[5.492676685891994, 4.370262152424703, 4.39392...","[6.709280191111111, 2.0382496900000002, 10.354...","[48.699919361787536, 44.77609028480145, 13.790...",19.315449,48.699919,8.461047,24.323674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,18,2000-01-03 02:29:06.741023+00:00,2000-01-05 01:32:30.067949+00:00,1 days 23:03:23.326926,3.077176,-24.392430,3.290386,-24.419062,5,"[5.149594524414461, 4.553605628962466, 5.08451...","[1.7412471688888889, 21.926795539444445, 10.36...","[21.422988453916386, 42.19822186694033, 19.675...",21.926796,42.198222,11.764120,24.190343
19,19,2000-01-03 03:49:13.509905+00:00,2000-01-04 14:20:43.116179+00:00,1 days 10:31:29.606274,4.573955,-21.768492,4.529423,-21.961956,3,"[4.424323615814165, 4.8063056873675825, 4.2949...","[14.6592333, 19.865657331666664]","[42.895473153531356, 11.088576724041175]",19.865657,42.895473,17.262445,26.992025


In [7]:
results[1]  # quakes

,Swarm ID,date,lat,lon,magnitude
0,<NA>,2000-01-01 13:12:52.944104+00:00,2.830076,-24.965474,4.007774
1,0,2000-01-01 13:35:33.740915+00:00,2.796269,-20.866927,4.716181
...,...,...,...,...,...
198,6,2000-01-05 16:20:15.778102+00:00,4.394290,-22.385421,4.407223
199,8,2000-01-05 16:37:38.729424+00:00,0.193753,-21.534493,5.001654


### 3. unpacking

In [8]:
df1, df2 = results
df1  # swarms

,Swarm ID,Start Time,End Time,Duration,Origin Latitude,Origin Longitude,Centre Latitude,Centre Longitude,Number of Earthquakes,Magnitudes,Time Intervals (h),Distance Intervals (km),Minimum Time Limit (h),Minimum Distance Limit (km),Average Time Interval (h),Average Distance Interval (km)
0,0,2000-01-01 13:35:33.740915+00:00,2000-01-05 10:52:01.417453+00:00,3 days 21:16:27.676538,2.796269,-20.866927,2.912540,-20.932767,10,"[4.716180922504579, 4.051831335693045, 4.71988...","[16.051896626666668, 5.807235363333334, 7.2911...","[37.50526709650672, 35.40305074737498, 24.0290...",26.103586,48.041596,10.363817,25.708724
1,1,2000-01-01 13:51:31.070540+00:00,2000-01-05 10:55:48.543430+00:00,3 days 21:04:17.472890,1.879701,-20.350824,1.976687,-20.367298,12,"[5.492676685891994, 4.370262152424703, 4.39392...","[6.709280191111111, 2.0382496900000002, 10.354...","[48.699919361787536, 44.77609028480145, 13.790...",19.315449,48.699919,8.461047,24.323674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,18,2000-01-03 02:29:06.741023+00:00,2000-01-05 01:32:30.067949+00:00,1 days 23:03:23.326926,3.077176,-24.392430,3.290386,-24.419062,5,"[5.149594524414461, 4.553605628962466, 5.08451...","[1.7412471688888889, 21.926795539444445, 10.36...","[21.422988453916386, 42.19822186694033, 19.675...",21.926796,42.198222,11.764120,24.190343
19,19,2000-01-03 03:49:13.509905+00:00,2000-01-04 14:20:43.116179+00:00,1 days 10:31:29.606274,4.573955,-21.768492,4.529423,-21.961956,3,"[4.424323615814165, 4.8063056873675825, 4.2949...","[14.6592333, 19.865657331666664]","[42.895473153531356, 11.088576724041175]",19.865657,42.895473,17.262445,26.992025


In [9]:
df2  # quakes

,Swarm ID,date,lat,lon,magnitude
0,<NA>,2000-01-01 13:12:52.944104+00:00,2.830076,-24.965474,4.007774
1,0,2000-01-01 13:35:33.740915+00:00,2.796269,-20.866927,4.716181
...,...,...,...,...,...
198,6,2000-01-05 16:20:15.778102+00:00,4.394290,-22.385421,4.407223
199,8,2000-01-05 16:37:38.729424+00:00,0.193753,-21.534493,5.001654


## Docstring

The docstring has information on the arguments that can be passed to the function `find_swarms()` and the values it returns. 

Arguments can be passed by position or keyword. Note that only the `eq_df` argument is required, while the rest have default values. However, the correct column labels must be passed to the function if the default values do not match column labels in the `eq_df` DataFrame.

In [10]:
help(swarms.find_swarms)

Help on function find_swarms in module swarms:

find_swarms(eq_df, time_lim=50, dist_lim=50, min_number=2, time_col='time', lat_col='latitude', lon_col='longitude', mag_col='mag')
    Group earthquakes into swarms and record swarm properties.
    
    Parameters
    ----------
    eq_df : pandas DataFrame
        DataFrame of earthquake data. Must have columns for origin time,
        latitude, longitude, and magnitude.
    time_lim : int or float, default 50
        Maximum time interval (in hours) between any given earthquake and
        the preceding earthquake in the same swarm.
    dist_lim : int or float, default 50
        Maximum distance interval (in kilometres) between any given
        earthquake and the nearest (in space) preceding earthquake in the
        same swarm.
    min_number : int, default 2
        Minimum number of earthquakes required to be grouped together to be
        considered a swarm.
    time_col : int or str, default "time"
        Label for the DataFram